# Stage 8A Optimization #1 — Cached + Batched \(\rho\)

Stage 8 profiling showed that bank preparation is dominated by spatial correlation.

This notebook changes **no physics**. It tests two implementation optimizations:

1. displacement cache is reused by array shape;
2. banks with the same antenna shape are evaluated as one GPU batch.

The existing Stage-2 Gauss–Hermite kernel remains unchanged.

The validation first compares the optimized path against the old dbar-based cache path, then benchmarks:
- old single-bank execution on a representative subset,
- optimized bucketed execution on the same subset,
- optimized execution on the entire 2000-bank scenario CSV.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

required = [
    'ris_gpu_geometry_lsp_stage67.py',
    'ris_gpu_physics_stage1.py',
    'ris_gpu_rho_stage2.py',
    'ris_gpu_environment_stage8_optimized.py',
]

for d in [ROOT,Path('/content')]:
    if str(d) not in sys.path:
        sys.path.insert(0,str(d))

missing = [
    name for name in required
    if not (ROOT/name).exists() and not (Path('/content')/name).exists()
]
assert not missing, "Eksik modüller:\n" + "\n".join(missing)

from ris_gpu_environment_stage8_optimized import (
    add_shape_key_columns,
    ArrayDisplacementCacheManager,
    validate_cached_batch_against_dbar_reference,
    benchmark_single_bank_reference,
    benchmark_bucketed_rho,
    auto_rho_chunk_size,
    estimate_rho_output_bytes_per_bank,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

In [ ]:
# Locate the scenario CSV.
candidates = [
    ROOT/'generate_train_scenarios_data_2000.csv',
    Path('/content/generate_train_scenarios_data_2000.csv'),
]

CSV = next((p for p in candidates if p.exists()),None)
assert CSV is not None, (
    "generate_train_scenarios_data_2000.csv dosyasını "
    "Drive RIS root'a veya /content altına koy."
)

df = pd.read_csv(CSV)

required_cols = [
    'fc','scenario_BR','scenario_RU',
    'ris_x','ris_y','ris_z',
    'gnb_x','gnb_y','gnb_z',
    'ue_x','ue_y','ue_z',
    'nT1','nT2','nR1','nR2','nRIS_x','nRIS_y',
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, missing

print("Rows:",len(df))
display(df.head())

In [ ]:
# Shape distribution and automatic chunk plan.
dsk = add_shape_key_columns(df)

shape_summary = (
    dsk.groupby('_shape_key')
       .size()
       .rename('banks')
       .reset_index()
)

shape_summary['chunk_fp32'] = shape_summary['_shape_key'].apply(
    lambda k: auto_rho_chunk_size(
        k,
        parity=False,
        target_memory_mb=768,
        safety_factor=3.0,
        max_batch=128,
    )
)

shape_summary['rho_MB_per_bank'] = shape_summary['_shape_key'].apply(
    lambda k: estimate_rho_output_bytes_per_bank(k,parity=False)/1024**2
)

shape_summary = shape_summary.sort_values(
    ['rho_MB_per_bank','banks'],
    ascending=[False,False]
).reset_index(drop=True)

print("Unique shape buckets:",len(shape_summary))
display(shape_summary)

## 1. Optimization correctness

Most common same-shape bucket is used for the initial test.

Optimized result:

\[
\text{shape cache}+\text{B-bank GPU call}
\]

Reference result:

\[
\text{bank-specific dbar cache}+\text{B=1 calls}.
\]

Double precision must remain at numerical-zero / machine-precision scale.

In [ ]:
most_common_key = (
    dsk.groupby('_shape_key')
       .size()
       .sort_values(ascending=False)
       .index[0]
)

same_shape = dsk[dsk['_shape_key']==most_common_key].drop(
    columns=['_shape_key']
).reset_index(drop=True)

print("Validation shape:",most_common_key)
print("Available rows:",len(same_shape))

val = validate_cached_batch_against_dbar_reference(
    same_shape,
    n_rows=min(4,len(same_shape)),
    device=device,
    parity=True,
    gh_pair_chunk=80,
)

display(
    pd.DataFrame(
        {'metric':list(val.keys()),'value':list(val.values())}
    )
)

rel = [v for k,v in val.items() if k.endswith('_relFro')]
worst = max(rel)

print("Worst optimized-vs-reference double rel error:",worst)
assert worst < 1e-12

print("PASS: cached/batched rho is numerically unchanged")

## 2. Fair subset speed comparison

To make the comparison useful, take a subset that contains several banks
from the most common shape buckets.

Baseline runs one bank at a time and rebuilds dbar caches.
Optimized path groups the same rows by shape and reuses normalized caches.

In [ ]:
# Build a representative subset with up to 8 rows from each of the
# 8 most common shape buckets.
top_keys = (
    dsk.groupby('_shape_key')
       .size()
       .sort_values(ascending=False)
       .head(8)
       .index
)

parts = []
for key in top_keys:
    parts.append(
        dsk[dsk['_shape_key']==key]
        .head(8)
        .drop(columns=['_shape_key'])
    )

subset = pd.concat(parts,ignore_index=True)
print("Representative subset banks:",len(subset))

In [ ]:
# Baseline; this intentionally does per-bank rho.
baseline = benchmark_single_bank_reference(
    subset,
    n_rows=len(subset),
    device=device,
    parity=False,
    gh_pair_chunk=80,
)

print("BASELINE")
print(json.dumps(baseline,indent=2))

In [ ]:
mgr = ArrayDisplacementCacheManager()

optimized_subset = benchmark_bucketed_rho(
    subset,
    device=device,
    parity=False,
    gh_pair_chunk=80,
    target_memory_mb=768,
    max_batch=128,
    safety_factor=3.0,
    cache_manager=mgr,
)

print("OPTIMIZED SUBSET")
for k,v in optimized_subset.items():
    if k != 'bucket_table':
        print(k,":",v)

display(optimized_subset['bucket_table'])

baseline_s_per_bank = baseline['rho_mean_seconds_per_bank']
optimized_s_per_bank = optimized_subset['seconds_per_bank']

speedup = baseline_s_per_bank / optimized_s_per_bank

print(
    f"\nSubset speedup: {speedup:.2f}x "
    f"({baseline_s_per_bank:.4f} -> "
    f"{optimized_s_per_bank:.4f} s/bank)"
)

## 3. Entire dataset benchmark

This streams the full CSV; the full rho tensors are **not retained**.
They are deleted after every shape/chunk exactly as a dataset generator
would consume/write features incrementally.

If GPU memory is tight, reduce `target_memory_mb` from 768 to 512 or 384.

In [ ]:
mgr_full = ArrayDisplacementCacheManager()

full = benchmark_bucketed_rho(
    df,
    device=device,
    parity=False,
    gh_pair_chunk=80,
    target_memory_mb=768,
    max_batch=128,
    safety_factor=3.0,
    cache_manager=mgr_full,
)

print("FULL DATASET")
for k,v in full.items():
    if k != 'bucket_table':
        print(k,":",v)

display(full['bucket_table'])

print(
    f"\nFull front-end rho throughput: "
    f"{full['banks_per_second']:.2f} bank/s"
)
print(
    f"Equivalent seconds per bank: "
    f"{full['seconds_per_bank']:.4f}"
)

In [ ]:
# Save benchmark tables for the future GitHub repo.
OUT_BUCKETS = Path('/content/stage8_rho_bucket_benchmark.csv')
OUT_SUMMARY = Path('/content/stage8_rho_optimization_summary.json')

full['bucket_table'].to_csv(OUT_BUCKETS,index=False)

summary = {
    'baseline_subset':baseline,
    'optimized_subset':{
        k:v for k,v in optimized_subset.items()
        if k != 'bucket_table'
    },
    'subset_speedup':float(speedup),
    'full_dataset':{
        k:v for k,v in full.items()
        if k != 'bucket_table'
    }
}

OUT_SUMMARY.write_text(
    json.dumps(summary,indent=2),
    encoding='utf-8'
)

print(OUT_BUCKETS)
print(OUT_SUMMARY)

## Sonraki karar

Bu benchmark bize iki şeyi söyleyecek:

1. shape batching/cache, mevcut \(\rho\) bottleneck'ini ne kadar düşürdü;
2. en yavaş kalan shape bucket hangisi.

Eğer \(\rho\) hâlâ açık ara bottleneck ise bir sonraki optimizasyon,
20 alpha-ray çağrısını daha fazla tensorize/fuse etmek olacaktır.

Yeterince hızlandıysa Stage 8B stochastic channel path'e geçeriz.